<div align="center">

  <a href="https://grayboxtech.github.io/weightslab/latest/index.html" target="_blank">
    <img width="100%" src="https://raw.githubusercontent.com/GrayboxTech/.github/main/profile/weightslab-banner-light.png" alt="WeightsLab banner"></a>

  <a href="https://github.com/GrayboxTech/weightslab/blob/main/LICENSE"><img src="https://img.shields.io/badge/License-Apache%202.0-blue.svg" alt="License"></a>
  <a href="https://github.com/GrayboxTech/weightslab/stargazers"><img src="https://img.shields.io/github/stars/GrayboxTech/weightslab?style=flat&color=5865F2" alt="Stars"></a>
  <a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?style=flat&color=5865F2&logo=pypi&logoColor=white" alt="Version"></a>
  <br>
  <a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/main/weightslab/examples/Notebooks/Usecases/ws-segmentation-loss-shapes.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open WeightsLab In Colab"></a>

  Welcome to the WeightsLab <b>custom decorated-signal</b> usecase! This notebook trains segmentation and adds a <b>user-defined, decorated</b> per-sample signal that classifies each sample's loss trajectory. See the <a href="https://grayboxtech.github.io/weightslab/latest/index.html">Docs</a> for details.</div>

# Segmentation + a custom decorated per-sample signal

This usecase trains the **BDD100k segmentation** example *in the notebook kernel* and layers on a
**user-defined signal** decorated with `@wl.signal`. The signal **subscribes** to the per-sample
segmentation loss, classifies each sample's loss **trajectory shape** (monotonic, plateaued,
spiked, ...), and writes the verdict back as a categorical tag - all live, per sample.

It also demonstrates the `min_step` parameter of `@wl.signal`: the classifier only starts firing
once each sample has enough history (here `min_step=505`).

> Why in-kernel (not `!python main.py`)? A decorated `@wl.signal` must be registered in the **same
> process** that runs training, so we build and train here rather than shelling out.

## Setup

Pick a **T4 GPU** runtime on Colab. We clone the repo (for the segmentation `utils/` and the
bundled `BDD_subset/` data) and install WeightsLab from the clone (needed for the `min_step`
parameter).

<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?color=5865F2&logo=pypi&logoColor=white" alt="PyPI - Version"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/dm/weightslab?color=5865F2" alt="PyPI - Downloads"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/pyversions/weightslab?color=5865F2&logo=python&logoColor=white" alt="PyPI - Python Version"></a>

In [ ]:
import os
if not os.path.isdir("weightslab"):
    !git clone --branch torch_collab_examples https://github.com/GrayboxTech/weightslab.git
%pip install -q ./weightslab
!pip install --upgrade "protobuf>=6.31.1"
%pip install "torchvision>=0.16"
%cd weightslab/weightslab/examples/PyTorch/ws-segmentation

## 1. Imports

We reuse the segmentation example's helper modules (`utils/`) for the dataset, model, and the
per-sample / per-instance Dice + BCE criterions.

In [ ]:
import numpy as np
import torch
from torch import optim
from tqdm.auto import tqdm

import weightslab as wl
from weightslab.components.global_monitoring import (
    guard_training_context, guard_testing_context,
)
from utils.data import BDD100kSegDataset, seg_collate
from utils.model import SmallUNet
from utils.criterions import (
    PerSampleDice, PerInstanceDice, PerSampleBCE, PerInstanceBCE,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Configuration

All tunables in one dict (like a `config.yaml`, with comments).

In [ ]:
config = {
    # -- Experiment ----------------------------------------------------------
    "experiment_name": "seg_loss_shapes",   # name shown in Weights Studio
    "root_log_dir": None,                     # None -> a temp dir is created

    # -- Data (bundled BDD_subset) -------------------------------------------
    "data_root": "BDD_subset",                # ships in the repo
    "num_classes": 6,                         # BDD label set
    "class_names": ["Background", "Ego Road", "Driveable Area",
                    "Lane Line 1", "Lane Line 2", "Lane Line 3"],
    "ignore_index": 255,                      # void pixels
    "image_size": 128,                        # images resized to image_size^2
    "max_train_samples": 256,                 # cap for a quick demo
    "max_test_samples": 100,
    "train_batch_size": 2,
    "test_batch_size": 2,

    # -- Optimizer -----------------------------------------------------------
    "learning_rate": 1e-3,

    # -- Training schedule ---------------------------------------------------
    "training_steps_to_do": 800,              # > min_step so the classifier fires
    "eval_full_to_train_steps_ratio": 100,    # full eval every N steps
    "write_export_ratio": 100,                # export history + dataframe every N steps

    # -- Custom decorated signal --------------------------------------------
    "shape_subscribe_to": "train_bce/sample", # per-sample loss the classifier watches
    "shape_compute_every_n_steps": 25,        # throttle the classifier
    "shape_min_step": 505,                    # WARM-UP: don't classify before this step

    # -- Services / live UI --------------------------------------------------
    "serving_grpc": True,
    "expose_ui": True,
    "backend_port": 50051,
}

## 3. Build data, model, optimizer, and the tracked seg signals

Same wrapping pattern as the segmentation example: dataloaders, model, optimizer, and the
per-sample / per-instance Dice (metric) + BCE (loss) signals are all passed through
`wl.watch_or_edit(...)`.

In [ ]:
train_ds = BDD100kSegDataset(root=config["data_root"], split="train",
                             num_classes=config["num_classes"], class_names=config["class_names"],
                             ignore_index=config["ignore_index"], image_size=config["image_size"],
                             max_samples=config["max_train_samples"])
val_ds = BDD100kSegDataset(root=config["data_root"], split="val",
                           num_classes=config["num_classes"], class_names=config["class_names"],
                           ignore_index=config["ignore_index"], image_size=config["image_size"],
                           max_samples=config["max_test_samples"])

wl.watch_or_edit(config, flag="hyperparameters", name=config["experiment_name"],
                 defaults=config, poll_interval=1.0)

train_loader = wl.watch_or_edit(
    train_ds, flag="data", loader_name="train_loader",
    batch_size=config["train_batch_size"], shuffle=True, is_training=True,
    compute_hash=False, array_autoload_arrays=False, array_return_proxies=True,
    array_use_cache=True, preload_labels=False, collate_fn=seg_collate)
test_loader = wl.watch_or_edit(
    val_ds, flag="data", loader_name="test_loader",
    batch_size=config["test_batch_size"], shuffle=False, is_training=False,
    compute_hash=False, array_autoload_arrays=False, array_return_proxies=True,
    array_use_cache=True, preload_labels=True, collate_fn=seg_collate)

model = wl.watch_or_edit(
    SmallUNet(in_channels=3, num_classes=config["num_classes"],
              image_size=config["image_size"]).to(device),
    flag="model", device=device, compute_dependencies=True)
optimizer = wl.watch_or_edit(
    optim.Adam(model.parameters(), lr=config["learning_rate"]), flag="optimizer")


def make_seg_signals(split):
    return {
        "dice_sample": wl.watch_or_edit(PerSampleDice(), flag="metric",
            name=f"{split}_dice/sample", per_sample=True, log=True),
        "dice_instance": wl.watch_or_edit(PerInstanceDice(), flag="metric",
            name=f"{split}_dice/instance", per_instance=True, log=True),
        "bce_sample": wl.watch_or_edit(PerSampleBCE(), flag="loss",
            name=f"{split}_bce/sample", per_sample=True, log=True),
        "bce_instance": wl.watch_or_edit(PerInstanceBCE(), flag="loss",
            name=f"{split}_bce/instance", per_instance=True, log=True),
    }

train_sig = make_seg_signals("train")
test_sig = make_seg_signals("test")

## 4. The custom, user-defined decorated signal

This is the point of the usecase. `classify_loss_shape` is **your** function, decorated with
`@wl.signal`. Because it sets `subscribe_to=<per-sample loss>`, WeightsLab calls it **once per
sample** each time that loss is logged. It pulls the sample's full loss history, classifies the
**shape** of the curve, and tags the sample.

Note `min_step=505`: the classifier is **skipped until training step 505**, so it only runs once
each sample has a meaningful trajectory. `compute_every_n_steps=25` then throttles it.

In [ ]:
# Allowed categorical values (registered so the UI shows all choices up front).
LOSS_SHAPE_LABELS = ["monotonic", "plateaued", "Flat_high",
                     "high_variance", "U_Shape", "Spiked"]
LOSS_SHAPE_CODES = {label: i for i, label in enumerate(LOSS_SHAPE_LABELS)}
wl.register_categorical_tag("loss_shape", LOSS_SHAPE_LABELS)

_MIN_HISTORY = 5

def _classify_loss_shape(values):
    """Classify a per-sample loss trajectory (scale-invariant thresholds)."""
    y = np.asarray(values, dtype=float)
    if y.size < _MIN_HISTORY:
        return None
    n = y.size
    first, last = float(y[0]), float(y[-1])
    ymin, ymax = float(y.min()), float(y.max())
    rng = max(ymax - ymin, 1e-8)
    mean = float(y.mean())
    cv = float(y.std()) / (abs(mean) + 1e-8)
    drop = (first - last) / (abs(first) + 1e-8)
    argmin = int(np.argmin(y))
    rebound = (last - ymin) / rng
    max_up_jump = float(np.diff(y).max()) / rng
    tail = y[int(0.6 * n):]
    tail_flat = (float(tail.std()) / (abs(float(tail.mean())) + 1e-8)) < 0.1
    if max_up_jump > 0.5:
        return "Spiked"
    if cv > 0.5:
        return "high_variance"
    if 0.2 * n < argmin < 0.8 * n and rebound > 0.3:
        return "U_Shape"
    if drop > 0.4:
        return "monotonic"
    if drop > 0.15 and tail_flat:
        return "plateaued"
    return "Flat_high"


@wl.signal(
    name="seg_loss_shape_classifier",
    subscribe_to=config["shape_subscribe_to"],
    compute_every_n_steps=config["shape_compute_every_n_steps"],
    min_step=config["shape_min_step"],   # <-- warm-up gate: skip until this step
    log=False,                            # side-effecting: we tag samples, no aggregate curve
)
def classify_loss_shape(ctx: wl.SignalContext) -> int:
    """Per-sample: classify the loss curve's shape and tag the sample."""
    history = wl.query_sample_history(
        ctx.sample_id, signal_name=config["shape_subscribe_to"],
        exp_hash=wl.get_current_experiment_hash())
    series = sorted(((step, val) for _, step, val, _ in history), key=lambda t: t[0])
    values = [v for _, v in series]
    label = _classify_loss_shape(values)
    if label is None:
        return -1
    wl.set_categorical_tag([ctx.sample_id], "loss_shape", label)
    return LOSS_SHAPE_CODES[label]

## 5. Train / eval steps

In [ ]:
def _run_signals(sig, outputs, labels, ids, preds):
    bce_sample = sig["bce_sample"](outputs, labels, batch_ids=ids, preds=preds)
    dice_sample = sig["dice_sample"](outputs, labels, batch_ids=ids)
    sig["dice_instance"](outputs, labels, batch_ids=ids)
    sig["bce_instance"](outputs, labels, batch_ids=ids)
    avg = 0.5 * dice_sample + 0.5 * bce_sample
    wl.save_signals({"combined_bce_dice_per_sample": avg}, ids)
    return avg, dice_sample


def train_step(loader, model, optimizer, sig):
    with guard_training_context:
        inputs, ids, labels, _ = next(loader)
        inputs = inputs.to(device)
        labels = [[m.to(device) for m in insts] for insts in labels]
        optimizer.zero_grad()
        outputs = model(inputs)
        preds = outputs.argmax(dim=1)
        loss_per_sample, _ = _run_signals(sig, outputs, labels, ids, preds)
        loss = loss_per_sample.mean()
        loss.backward()
        optimizer.step()
    return float(loss.detach().cpu().item())


def evaluate(loader, model, sig, n_batches):
    losses = dices = 0.0
    with guard_testing_context, torch.no_grad():
        for inputs, ids, labels, _ in loader:
            inputs = inputs.to(device)
            labels = [[m.to(device) for m in insts] for insts in labels]
            outputs = model(inputs)
            preds = outputs.argmax(dim=1)
            loss_per_sample, dice_sample = _run_signals(sig, outputs, labels, ids, preds)
            losses += torch.mean(loss_per_sample)
            dices += torch.mean(dice_sample)
    return float((losses / n_batches)), float((dices / n_batches) * 100.0)

## 6. (Optional) Expose the backend for the live UI

Run this to open a raw-TCP [`bore`](https://github.com/ekzhang/bore) tunnel (`bore.pub`, no signup)
so a Weights Studio on your machine can connect. In Studio you'll see samples getting tagged with
their `loss_shape` **after step 505**.

In [ ]:
import os, re, tarfile, threading, urllib.request, subprocess

endpoint = None
if config["expose_ui"]:
    bore = os.path.join(os.getcwd(), "bore")
    if not os.path.exists(bore):
        BORE = "v0.6.0"
        urllib.request.urlretrieve(
            f"https://github.com/ekzhang/bore/releases/download/{BORE}/bore-{BORE}-x86_64-unknown-linux-musl.tar.gz",
            "bore.tar.gz")
        with tarfile.open("bore.tar.gz") as t:
            t.extractall()
        os.chmod(bore, 0o755)
    proc = subprocess.Popen([bore, "local", str(config["backend_port"]), "--to", "bore.pub"],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        m = re.search(r"bore\.pub:(\d+)", line)
        if m:
            endpoint = f"bore.pub:{m.group(1)}"
            break
    threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()
    print("=" * 60)
    print("Backend exposed at:", endpoint)
    print("On your machine: weightslab ui launch  &&  weightslab tunnel", endpoint)
    print("=" * 60)
else:
    print("expose_ui is False - no tunnel.")

## 7. Serve and train

Watch the progress bar: before step 505 the `seg_loss_shape_classifier` stays silent; once the
warm-up gate opens it starts tagging samples by loss shape (visible in Studio and in the exported
dataframe).

In [ ]:
wl.serve(serving_grpc=config["serving_grpc"])
wl.start_training(timeout=3)

steps = config["training_steps_to_do"]
eval_ratio = config["eval_full_to_train_steps_ratio"]
export_ratio = config["write_export_ratio"]

test_loss = test_dice = None
pbar = tqdm(range(steps), desc="Training")
for step in pbar:
    age = model.get_age() if hasattr(model, "get_age") else step
    tr_loss = train_step(train_loader, model, optimizer, train_sig)
    if age > 0 and age % eval_ratio == 0:
        test_loss, test_dice = evaluate(test_loader, model, test_sig, len(test_loader))
    if age > 0 and age % export_ratio == 0:
        wl.write_history(); wl.write_dataframe()
    post = {"loss": f"{tr_loss:.3f}", "shapes": "on" if age >= config["shape_min_step"] else "warmup"}
    if test_dice is not None:
        post["dice"] = f"{test_dice:.1f}%"
    pbar.set_postfix(post)

wl.write_history(); wl.write_dataframe()
print("Done.")

## 8. Inspect the loss-shape tags

After step 505 the classifier tags each sample. Export shows the `tag:loss_shape` column and the
numeric `seg_loss_shape_classifier` signal - group or sort by them in Studio to triage samples by
how their loss behaved.

In [ ]:
import glob, pandas as pd
lg = wl.write_dataframe()  # returns the written path
paths = sorted(glob.glob(os.path.join(os.path.dirname(lg), "*dataframe*.json")), key=os.path.getmtime)
if paths:
    df = pd.read_json(paths[-1])
    shape_cols = [c for c in df.columns if "loss_shape" in c.lower()]
    print("shape-related columns:", shape_cols)
    display(df[shape_cols].head(30) if shape_cols else df.head())
else:
    print("No export found yet.")

## See it live in Weights Studio

**Colab has no Docker daemon**, so run Studio on your own machine and point it at this backend with
the `bore.pub:<port>` printed in Section 6:

```bash
pip install weightslab
weightslab ui launch                       # Terminal 1 -> http://localhost:5173
weightslab tunnel bore.pub:12345           # Terminal 2 -> host:port from Section 6
```

Group the sample grid by the `loss_shape` tag to see which samples plateaued, spiked, or never
learned - the decorated signal wrote those verdicts live, starting at step 505.

---

<div align="center">
Crafted by <a href="https://github.com/GrayboxTech/weightslab">GrayboxTech</a> - if WeightsLab helps you catch a bad label, drop us a star on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a>.
</div>